<a href="https://www.kaggle.com/code/dulapurkaystha/snack-guardian-ai?scriptVersionId=280189751" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Snack Guardian AI: A Multi-Agent Gut-Friendly Snack Assistant

In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


In [2]:
from google.adk.agents import Agent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("✅ ADK components imported successfully.")

retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

model = Gemini(
    model_name="gemini-2.5-flash-lite",
    retry_options=retry_config
)

✅ ADK components imported successfully.


In [3]:
import json
from typing import Dict, Any

# Very simple in-memory "database"
user_profiles: Dict[str, Dict[str, Any]] = {
    "default_user": {
        "name": ["Dula"],
        "gut_conditions": ["GERD"],
        "diet_preferences": ["vegetarian", "no onion", "no garlic"],
        "known_triggers": ["tomato", "citrus", "paprika"],
        "safe_foods": ["peanut butter", "banana", "rice"],
        "likes": ["warm", "gentle"],
        "dislikes": ["mint"],
        "never_suggest": [],
        "notes": "Starter profile for testing."
    }
}
print("✅ Profile created")

✅ Profile created


In [4]:
def get_user_profile(user_id:str) -> Dict[str, Any]:
    """
    Return the stored gut and snack profile for a given user.
    
    Args:
        user_id: A string ID for the user (e.g. "default_user")
    
    Returns:
        A dictionary containing gut conditions, triggers, safe foods, likes, dislikes, etc.
        If the user profile is not found, returns an empty dictionary.
    
    """
    return user_profiles.get(user_id,{})

# DEBUG
# print("Test: get_user_profile('default_user'):\n")
# print(get_user_profile("default_user"))

def update_user_profile(user_id:str, key: str, value: str) -> Dict[str, Any]:
    """
    Updates the user's gut/snack profile by appending a value under a key
    This tool is used to remember new triggers, safe foods, likes or dislikes.

    Args:
        user_id: A string ID for the user (e.g. "default_user").
        key: The field to update (e.g. "known_triggers", "safe_foods" etc)
        value: The new item to add to the list for that key (e.g. "popcorn")

    Returns:
        The updated profile dictionary for that user
    """
    print(f"update_user_profile called with user_id={user_id}, key={key}, value={value}")
    profile = user_profiles.setdefault(user_id, {})
    field = profile.setdefault(key, [])
    if isinstance(field, list) and value not in field:
        field.append(value)
    profile[key] = field
    return profile

# DEBUG
# print("✅ update_user_profile tool created")
# update_user_profile("default_user", "known_triggers", "popcorn")
# print(get_user_profile("default_user"))

In [5]:
root_agent = Agent(
    name="helpful_snack_agent",
    model=model,
    # description="A simple snack agent that can suggest snacks."
    instruction="""
    You are a helpful assistant.
        
    - When the user tells you about a new trigger, safe food, preference,
      or something they never want suggested again, call
      update_user_profile(user_id="default_user", key=..., value=...).

    - Use these keys when appropriate:
        - "known_triggers" for foods that cause problems (e.g. popcorn)
        - "safe_foods" for foods that feel good
        - "likes" for textures/flavors they enjoy (e.g. warm, crunchy)
        - "dislikes" for things they don't like
        - "never_suggest" for things never to recommend

    - When the user asks what you know about them, call
      get_user_profile(user_id="default_user") and summarize.

    Always use these tools instead of guessing.
    """,
    tools=[get_user_profile, update_user_profile],
)

print("✅ Root Agent defined.")

✅ Root Agent defined.


In [6]:
runner = InMemoryRunner(agent=root_agent)

print("✅ Runner created.")

✅ Runner created.


In [7]:
# response = await runner.run_debug(
#     "I have mild acid reflux. Can you suggest a gentle evening snack?"
# )

response = await runner.run_debug(
    "Remember that I cannot eat spicy foods."
)

response = await runner.run_debug(
    "What do you know about my gut triggers now?",
)

print(get_user_profile("default_user"))


 ### Created new session: debug_session_id

User > Remember that I cannot eat spicy foods.


update_user_profile called with user_id=default_user, key=dislikes, value=spicy foods
helpful_snack_agent > Okay, I've noted that you dislike spicy foods. I won't recommend them in the future.

 ### Continue session: debug_session_id

User > What do you know about my gut triggers now?


helpful_snack_agent > You've mentioned that your known triggers include tomato, citrus, and paprika.
{'name': ['Dula'], 'gut_conditions': ['GERD'], 'diet_preferences': ['vegetarian', 'no onion', 'no garlic'], 'known_triggers': ['tomato', 'citrus', 'paprika'], 'safe_foods': ['peanut butter', 'banana', 'rice'], 'likes': ['warm', 'gentle'], 'dislikes': ['mint', 'spicy foods'], 'never_suggest': [], 'notes': 'Starter profile for testing.'}
